<a href="https://colab.research.google.com/github/lilhawkeye2002-ux/Whisper_Notebooks/blob/main/Whisper_Bulk_Transcriber_AltTimestampMethod_FasterWhisper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ Whisper Bulk Transcriber — Timestamped Transcript Variant (faster-whisper)

Transcribe audio and video files into **millisecond-accurate, timestamped** plain text using OpenAI's Whisper `large-v2` model via the [faster-whisper](https://github.com/SYSTRAN/faster-whisper) CTranslate2 backend.

faster-whisper is 2–4× faster than the standard openai-whisper library on T4 GPU, with equivalent or better transcription quality.

---

## What makes this different

| | Standard `Whisper_Bulk_Transcriber` | This notebook |
|---|---|---|
| `.txt` content | Plain text, no timestamps | `[HH:MM:SS.mmm --> HH:MM:SS.mmm]  spoken text` |
| Timestamp source | Model token predictions (20 ms steps) | CTranslate2 forced alignment on word boundaries |
| How accurate | ±20–60 ms | Word-boundary aligned to actual audio |
| `.srt` / `.vtt` | Included | Included (also word-aligned) |
| Backend | PyTorch (openai-whisper) | CTranslate2 (faster-whisper) |
| Speed on T4 | Baseline | 2–4× faster |

### How the timestamps work

Enabling `word_timestamps=True` triggers CTranslate2’s forced alignment algorithm, which observes which audio frames the decoder attended to when producing each word and overwrites each segment’s `start`/`end` with word-boundary-aligned float values. This is functionally equivalent to openai-whisper’s DTW approach — measurement-based alignment, not prediction.

---

## Performance Tuning Flags

At the top of the code cell you can toggle five flags:

| Flag | Default | What it does | Typical gain |
|---|---|---|---|
| `PRECONVERT_AUDIO` | `True` | Pre-converts every file to 16 kHz mono WAV before transcription, eliminating per-chunk resampling overhead. Especially helpful for MP3/AAC/M4A inputs. | 10–30% |
| `COMPUTE_TYPE` | `"auto"` | Inference precision: `float16` (FP16, ~3–4 GB VRAM), `int8_float16` (~30% less VRAM, minimal quality loss), `int8` (CPU INT8). `"auto"` selects `float16` on GPU and `int8` on CPU. | varies |
| `VAD_FILTER` | `True` | Built-in silence detection using bundled silero-vad. Skips non-speech regions entirely before transcription. No extra install. | 20–70% |
| `CPU_THREADS` | `4` | CTranslate2 intra-op thread count for CPU inference. `0` = auto-detect. | small |
| `FAST_DECODE` | `False` | Switches to greedy decoding (`beam_size=1`, `temperature=0`) with fallback retries disabled. Much faster on clean audio; not recommended for noisy recordings. | up to 2× |

### Expected VRAM usage on T4

| Compute type | VRAM (model + activations) |
|---|---|
| `float16` | ~3–4 GB |
| `int8_float16` | ~2–3 GB |
| `float32` | ~6–8 GB — **avoid on T4** |

> `COMPUTE_TYPE="auto"` selects `float16` on GPU automatically.

---

## How To Use (3 Steps)

### Step 1 — Enable a free GPU
**Runtime → Change runtime type → Hardware accelerator → T4 GPU**

### Step 2 — Run the cell for the first time
Click **▶** on the code cell below. It will create the upload folder and stop with:
> *"Directory created. Upload your audio/video files there, then run this cell again."*

### Step 3 — Upload files and run again
1. Open the **Files panel** (📁 in the left sidebar)
2. Navigate to `/content/bulk_process_audios_here`
3. **Drag and drop** your audio or video files in
4. Click **▶** again — transcription starts automatically

> **Re-running the cell** is safe — if the model is already loaded with the same `COMPUTE_TYPE`, it will be reused without reloading.

---

## Output

For each source file, three outputs are saved to `/content/audio_transcription/`:

| Format | Content |
|---|---|
| `.txt` | `[00:00:00.000 --> 00:00:04.820]  Welcome everyone, today we're going to...` |
| `.srt` | Subtitle format with word-aligned timestamps |
| `.vtt` | Web caption format with word-aligned timestamps |

All files are zipped to `/content/all_transcribed_files.zip` — **download before closing the session**.

Per file, the cell also prints:
- Detected language and confidence (e.g. `Language: en (99%)`)
- VAD stats showing how much audio was skipped (e.g. `VAD: 120.0s → 87.3s (27% filtered)`)
- Elapsed transcription time

---

## Supported formats
| Audio | Video |
|-------|-------|
| `.mp3` `.wav` `.m4a` `.flac` | `.mp4` `.mov` `.avi` `.mkv` |
| `.aac` `.ogg` `.wma` `.opus` | `.webm` |
| `.aiff` `.amr` `.au` | |

> 🎬 Video files are handled automatically — the audio track is extracted and resampled before transcription.

---

## Troubleshooting

| Issue | Fix |
|---|---|
| Transcription very slow | Switch to GPU: **Runtime → Change runtime type → T4 GPU** |
| `ffmpeg` not found | Add a cell above: `!apt install -y ffmpeg` |
| `CUDA out of memory` on re-run | Model is already loaded — this is now handled automatically. If it persists, do **Runtime → Restart session** and re-run. |
| GPU out of memory (first run) | Set `COMPUTE_TYPE = "int8_float16"` to reduce VRAM, or do **Runtime → Restart session** and re-run |
| File skipped `[SKIP]` | File is empty (0 bytes) or unreadable — re-upload it |
| Session expired before download | Re-run the cell — all files are re-transcribed and re-zipped |

In [ ]:
# @title ## Whisper Bulk Transcriber — Timestamped (faster-whisper)
# @markdown ### Outputs `.txt` with word-aligned millisecond timestamps, plus `.srt` and `.vtt`.
# @markdown Final zip saved to `/content/all_transcribed_files.zip`.
# @markdown **Re-running this cell is safe** — the model is reused if already loaded.

# ── Phase 0: Constants & Performance Tuning Flags ────────────────────────────
# Only stdlib modules needed here — no pip installs required for directory check.

import os
import sys

BULK_DIR   = "/content/bulk_process_audios_here"
OUTPUT_DIR = "/content/audio_transcription"
ZIP_PATH   = "/content/all_transcribed_files.zip"
USE_MODEL  = "large-v2"

VIDEO_EXTENSIONS = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}

SUPPORTED_EXTENSIONS = {
    '.mp3', '.wav', '.m4a', '.flac', '.aac', '.ogg', '.wma',
    '.aiff', '.opus', '.amr', '.au',
} | VIDEO_EXTENSIONS

TRANSCRIBED_EXTS = {'.txt', '.srt', '.vtt'}

# ── Performance Tuning ────────────────────────────────────────────────────────
# Adjust these flags before running to trade speed against quality.

# Pre-convert every audio file to 16 kHz mono WAV before passing it to Whisper.
# Eliminates internal resampling overhead on every 30-second chunk. Especially
# impactful for compressed formats (MP3, AAC, M4A). WAV inputs already at
# 16 kHz mono are detected and passed through unchanged. Recommended: True.
PRECONVERT_AUDIO = True  # @param {type:"boolean"}

# Inference precision. "auto" selects float16 on GPU and int8 on CPU.
# float16       : FP16 — ~3-4 GB VRAM on T4, fastest GPU mode.
# int8_float16  : INT8 weights + FP16 activations — ~30% less VRAM, minimal quality loss.
# int8          : INT8 — CPU only, much faster than float32.
# float32       : Full precision — avoid on T4 (uses 2x VRAM vs float16).
COMPUTE_TYPE = "auto"  # @param ["auto", "float16", "int8_float16", "int8", "float32"]

# Built-in silence filter using bundled silero-vad. Detects and skips non-speech
# regions before transcription. No extra install required.
# Typical speedup: 20-70% on recordings with silence, music, or long pauses.
VAD_FILTER   = True  # @param {type:"boolean"}

# CTranslate2 intra-op CPU thread count. 0 = auto-detect.
# Only matters when DEVICE is cpu. Ignored on GPU.
CPU_THREADS  = 4  # @param {type:"integer"}

# Fast decode mode: greedy decoding (beam_size=1, temperature=0), fallback
# retries disabled. Up to 2x faster on clean speech. Not recommended for
# noisy, heavily accented, or poor-quality audio.
FAST_DECODE  = False  # @param {type:"boolean"}

# ── Phase 1: Directory Check (runs FIRST — before any installs) ───────────────
# Instant and dependency-free. Saves the user from waiting through pip installs
# and model loading when there is nothing to do yet.

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(BULK_DIR):
    os.makedirs(BULK_DIR)
    sys.exit(
        f"\nDirectory created: '{BULK_DIR}'\n"
        f"Upload your audio/video files there, then run this cell again.\n"
    )

_all_in_dir = [
    os.path.join(BULK_DIR, f) for f in os.listdir(BULK_DIR)
    if os.path.isfile(os.path.join(BULK_DIR, f))
]
audio_files = sorted([
    f for f in _all_in_dir
    if os.path.splitext(f)[1].lower() in SUPPORTED_EXTENSIONS
])

if not audio_files:
    sys.exit(
        f"\nNo supported audio/video files found in '{BULK_DIR}'.\n"
        f"Supported formats: {', '.join(sorted(SUPPORTED_EXTENSIONS))}\n"
        f"Upload files there and run this cell again.\n"
    )

print(f"Found {len(audio_files)} file(s) to transcribe. Loading dependencies...")

# ── Phase 2: Full Imports ─────────────────────────────────────────────────────
# Only reached when files are confirmed present.

import subprocess
import tempfile
import time
import unicodedata
import zipfile

# ── Phase 3: Library Check / Install ─────────────────────────────────────────
# faster-whisper installs ctranslate2 (CTranslate2 backend), huggingface_hub,
# tokenizers, and silero-vad (for VAD_FILTER) as transitive dependencies.
# No torch install required — faster-whisper uses CTranslate2 directly.

def _install_libraries():
    cmd = [
        sys.executable, "-m", "pip", "install", "-q",
        "--root-user-action=ignore",
        "faster-whisper",
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("[ERROR] pip install failed:")
        print(result.stderr)
        sys.exit(1)

try:
    from faster_whisper import WhisperModel
    import ctranslate2
    print("Libraries already installed. Skipping installation.")
except ImportError:
    print("Installing required libraries...")
    _install_libraries()
    try:
        from faster_whisper import WhisperModel
        import ctranslate2
        print("Libraries installed and imported successfully.")
    except ImportError as e:
        print(f"[ERROR] Import failed after installation: {e}")
        sys.exit(1)

# ── Phase 4: ffmpeg Binary Pre-flight Check ───────────────────────────────────

_ffmpeg_check = subprocess.run(
    ["ffmpeg", "-version"], capture_output=True
)
if _ffmpeg_check.returncode != 0:
    print("[ERROR] ffmpeg binary not found on PATH.")
    print("Fix: add a cell above with   !apt install -y ffmpeg   and run it first.")
    sys.exit(1)
print("ffmpeg binary confirmed.")

# ── Phase 4b: Environment Diagnostics ────────────────────────────────────────
# CTranslate2 (not torch) is used for device detection.
# GPU name is queried via nvidia-smi since torch is not a dependency.

DEVICE = "cuda" if ctranslate2.get_cuda_device_count() > 0 else "cpu"
print(f"Device         : {DEVICE}")
if DEVICE == "cuda":
    _smi_name = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True, text=True,
    )
    if _smi_name.returncode == 0:
        print(f"GPU            : {_smi_name.stdout.strip()}")
print(f"CTranslate2    : {ctranslate2.__version__}")

# Resolve COMPUTE_TYPE: "auto" maps to float16 on GPU, int8 on CPU.
_compute_type = (
    ("float16" if DEVICE == "cuda" else "int8") if COMPUTE_TYPE == "auto"
    else COMPUTE_TYPE
)
print(f"Compute type   : {_compute_type}")

# ── Phase 5: Per-file Pre-validation ─────────────────────────────────────────

valid_audio_files = []
for _f in audio_files:
    _name = os.path.basename(_f)
    if not os.access(_f, os.R_OK):
        print(f"[SKIP] Not readable (check permissions): {_name}")
        continue
    if os.path.getsize(_f) == 0:
        print(f"[SKIP] Empty file (0 bytes): {_name}")
        continue
    valid_audio_files.append(_f)

if not valid_audio_files:
    sys.exit("[ERROR] No valid audio files remain after validation. Aborting.")

audio_files = valid_audio_files
print(f"{len(audio_files)} file(s) passed validation.")

# ── Phase 6: Model Load ───────────────────────────────────────────────────────

from faster_whisper.utils import format_timestamp  # noqa: E402
# Note: faster-whisper has no get_writer — SRT/VTT are written by custom
# functions in Phase 8.

if DEVICE == "cpu":
    _total_secs = 0.0
    _duration_known = True
    for _af in audio_files:
        _probe = subprocess.run(
            [
                "ffprobe", "-v", "error",
                "-show_entries", "format=duration",
                "-of", "default=noprint_wrappers=1:nokey=1",
                _af,
            ],
            capture_output=True, text=True,
        )
        try:
            _total_secs += float(_probe.stdout.strip())
        except ValueError:
            _duration_known = False
            break

    _EST_SLOWDOWN = 10  # faster-whisper INT8 on CPU is ~2x faster than openai-whisper
    _border = "!" * 64
    print(f"\n{_border}")
    print("  WARNING: NO GPU DETECTED — TRANSCRIPTION WILL BE")
    print("  OVERWHELMINGLY SLOW (roughly 1 transcribed line per minute).")
    print()
    if _duration_known and _total_secs > 0:
        _audio_mins = _total_secs / 60
        _est_mins   = (_total_secs * _EST_SLOWDOWN) / 60
        _est_hrs    = _est_mins / 60
        print(f"  Total audio duration : ~{_audio_mins:.0f} min")
        if _est_hrs >= 1:
            print(f"  Estimated CPU time   : ~{_est_hrs:.1f} hours  (could be longer)")
        else:
            print(f"  Estimated CPU time   : ~{_est_mins:.0f} minutes  (could be longer)")
        print()
    print("  STRONGLY RECOMMENDED: switch to a GPU runtime first.")
    print("  Runtime > Change runtime type > Hardware accelerator > T4 GPU")
    print(f"{_border}\n")

    _answer = input(
        "Continue on CPU anyway? This will take an extremely long time. [y/N]: "
    ).strip().lower()

    if _answer != "y":
        sys.exit(
            "\nAborted. To enable GPU:\n"
            "  Runtime > Change runtime type > Hardware accelerator > T4 GPU\n"
            "Then run this cell again.\n"
        )
    print("Acknowledged. Continuing on CPU — this will take a very long time.\n")

# Guard: skip load if model already resident with matching device + compute_type.
# faster-whisper's WhisperModel has no .parameters() like a PyTorch model, so
# a sentinel variable tracks what is currently loaded.
_need_load = True
_target_model_key = (USE_MODEL, DEVICE, _compute_type)
if (
    "model" in globals()
    and isinstance(model, WhisperModel)
    and globals().get("_loaded_model_key") == _target_model_key
):
    print(f"Model '{USE_MODEL}' already loaded on {DEVICE} ({_compute_type}). Reusing.")
    _need_load = False

if _need_load:
    print(f"Loading '{USE_MODEL}' model on {DEVICE} (compute_type={_compute_type})...")
    try:
        model = WhisperModel(
            USE_MODEL,
            device=DEVICE,
            compute_type=_compute_type,
            cpu_threads=CPU_THREADS,
        )
        _loaded_model_key = _target_model_key
        print(f"Model '{USE_MODEL}' loaded.")
    except Exception as e:
        print(f"[ERROR] Failed to load model: {e}")
        sys.exit(1)

# VRAM via nvidia-smi — CTranslate2 allocates GPU memory outside PyTorch's
# memory manager, so torch.cuda.memory_allocated() would show 0.
if DEVICE == "cuda":
    _smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.used,memory.total",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True,
    )
    if _smi.returncode == 0:
        _p = _smi.stdout.strip().split(",")
        print(
            f"VRAM: {int(_p[0].strip())/1024:.2f} GB used "
            f"/ {int(_p[1].strip())/1024:.2f} GB total"
        )

# ── Phase 7: Audio Preparation Helper ────────────────────────────────────────
#
# Converts any audio/video file to a 16 kHz mono PCM WAV before transcription.
#
# Why 16 kHz mono WAV?
#   faster-whisper internally resamples audio to 16 kHz mono via ffmpeg.
#   Doing this once upfront eliminates repeated resampling overhead on each
#   30-second chunk. Especially impactful for compressed formats (MP3, AAC, M4A)
#   where decoding is CPU-bound and Colab CPUs are slow relative to the T4 GPU.
#
# Optimisation path:
#   video file  → strip video stream + resample to 16 kHz mono WAV
#   audio file  → resample to 16 kHz mono WAV  (if PRECONVERT_AUDIO=True)
#   .wav file already at 16 kHz mono → returned as-is (no conversion)

def _is_already_optimal_wav(path: str) -> bool:
    "Return True if the file is already a 16 kHz mono PCM WAV (no conversion needed)."
    if os.path.splitext(path)[1].lower() != ".wav":
        return False
    _probe = subprocess.run(
        [
            "ffprobe", "-v", "error",
            "-select_streams", "a:0",
            "-show_entries", "stream=codec_name,sample_rate,channels",
            "-of", "default=noprint_wrappers=1:nokey=1",
            path,
        ],
        capture_output=True, text=True,
    )
    info = _probe.stdout.strip().split("\n")
    return (
        len(info) >= 3
        and info[0].strip() == "pcm_s16le"
        and info[1].strip() == "16000"
        and info[2].strip() == "1"
    )


def _prepare_audio(source_path: str, is_video: bool) -> tuple:
    # Convert source_path to a 16 kHz mono WAV suitable for faster-whisper.
    # Returns (tmp_path, converted) where:
    #   tmp_path  - path to the WAV file to pass to model.transcribe()
    #   converted - True if a temp file was created (must be deleted after use)
    # If the source is already an optimal WAV, returns (source_path, False).
    if not is_video and _is_already_optimal_wav(source_path):
        return source_path, False

    tmp = tempfile.NamedTemporaryFile(
        suffix=".wav",
        prefix="_whisper_pcm_",
        delete=False,
    )
    tmp_path = tmp.name
    tmp.close()

    cmd = [
        "ffmpeg", "-y",
        "-i", source_path,
        "-vn",                   # discard video stream (no-op for pure audio)
        "-acodec", "pcm_s16le",  # 16-bit signed PCM
        "-ar", "16000",          # 16 kHz sample rate
        "-ac", "1",              # mono
        tmp_path,
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        try:
            os.unlink(tmp_path)
        except OSError:
            pass
        err = proc.stderr.strip().splitlines()[-1] if proc.stderr.strip() else "unknown ffmpeg error"
        raise RuntimeError(f"Audio preparation failed: {err}")

    return tmp_path, True


# ── Phase 8: Output Writers ───────────────────────────────────────────────────
#
# faster-whisper has no built-in SRT/VTT writers (unlike openai-whisper's
# get_writer). All three formats are implemented here using
# faster_whisper.utils.format_timestamp(seconds, always_include_hours, decimal_marker).
#
# Timestamp alignment:
#   With word_timestamps=True, CTranslate2's forced alignment overwrites each
#   segment's .start/.end with word-boundary-aligned values — equivalent in
#   precision to openai-whisper's DTW approach. The .txt output format is
#   identical to the openai-whisper AltTimestampMethod variant.

def _write_timestamped_txt(segments: list, output_path: str) -> None:
    "Write HH:MM:SS.mmm timestamped transcript from faster-whisper segments."
    with open(output_path, "w", encoding="utf-8") as f:
        for seg in segments:
            start = format_timestamp(seg.start, always_include_hours=True)
            end   = format_timestamp(seg.end,   always_include_hours=True)
            text  = seg.text.strip()
            if text:
                f.write(f"[{start} --> {end}]  {text}\n")


def _write_srt(segments: list, output_path: str) -> None:
    "Write SubRip (.srt) subtitle file from faster-whisper segments."
    with open(output_path, "w", encoding="utf-8") as f:
        idx = 1
        for seg in segments:
            text = seg.text.strip()
            if not text:
                continue
            # SRT requires comma as decimal separator: 00:00:04,820
            start = format_timestamp(seg.start, always_include_hours=True, decimal_marker=",")
            end   = format_timestamp(seg.end,   always_include_hours=True, decimal_marker=",")
            f.write(f"{idx}\n{start} --> {end}\n{text}\n\n")
            idx += 1


def _write_vtt(segments: list, output_path: str) -> None:
    "Write WebVTT (.vtt) subtitle file from faster-whisper segments."
    with open(output_path, "w", encoding="utf-8") as f:
        f.write("WEBVTT\n\n")
        for seg in segments:
            text = seg.text.strip()
            if not text:
                continue
            # VTT spec rejects a first cue that starts at exactly 00:00:00.000
            start_s = max(seg.start, 0.001)
            start = format_timestamp(start_s, always_include_hours=True)
            end   = format_timestamp(seg.end,  always_include_hours=True)
            f.write(f"{start} --> {end}\n{text}\n\n")


# ── Phase 9: Transcription Options ───────────────────────────────────────────
#
# Two presets controlled by FAST_DECODE. Key differences from openai-whisper:
#   - No "fp16" key — precision is set by compute_type at model load time.
#   - No "word_timestamps" key — passed directly to model.transcribe().
#   - "log_prob_threshold" (not "logprob_threshold" as in openai-whisper).
#   - "vad_filter" key enables built-in silero-vad silence detection.
#   - temperature accepts a list (fallback schedule) or a single float.
#
#   FAST_DECODE=False (default — quality mode):
#     beam_size=5 activates beam search. best_of=5 is ignored when beam_size>1
#     (faster-whisper, like openai-whisper, uses beam search and disables
#     stochastic sampling when beam_size>1). temperature is a 6-step fallback
#     retry schedule triggered when a segment fails quality checks.
#
#   FAST_DECODE=True (speed mode):
#     Greedy decoding (beam_size=1, temperature=0.0). Fallback retries disabled.
#     Up to 2x faster on clean speech. Not for noisy audio.

if FAST_DECODE:
    _options = {
        "beam_size":                   1,
        "temperature":                 0.0,
        # Disable fallback retries — prevents repeated decode passes per segment.
        "compression_ratio_threshold": None,
        "log_prob_threshold":          None,
        "no_speech_threshold":         0.6,
        "condition_on_previous_text":  True,
        "initial_prompt":              None,
        "language":                    None,
        "task":                        "transcribe",
        "vad_filter":                  VAD_FILTER,
    }
    print("FAST_DECODE mode: greedy decoding, fallback retries disabled.")
else:
    _options = {
        # beam_size=5: beam search — primary quality lever.
        # Set to 1 for greedy decoding (~5x faster decode, slight quality drop).
        "beam_size":                   5,
        # best_of=5: ignored when beam_size > 1 (beam search active).
        # Kept for compatibility if beam_size is later set to 1.
        "best_of":                     5,
        # temperature: fallback retry schedule. Whisper retries a segment with
        # the next temperature if it fails compression_ratio or log_prob checks.
        "temperature":                 [0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
        "condition_on_previous_text":  True,
        "initial_prompt":              None,
        "language":                    None,
        "task":                        "transcribe",
        "no_speech_threshold":         0.4,
        "log_prob_threshold":          -1.5,
        "compression_ratio_threshold": 2.2,
        "vad_filter":                  VAD_FILTER,
    }

_mode_label = "FAST (greedy)" if FAST_DECODE else "QUALITY (beam search)"
print(f"Decode mode    : {_mode_label}")
print(f"Audio preconv  : {'enabled' if PRECONVERT_AUDIO else 'disabled'}")
print(f"VAD filter     : {'enabled' if VAD_FILTER else 'disabled'}")

# ── Phase 10: Transcription Loop ─────────────────────────────────────────────
#
# Key difference from openai-whisper version:
#   model.transcribe() returns a *lazy generator* of Segment objects, not a
#   result dict. Transcription runs as the generator is iterated. Consuming it
#   into a list gives real-time segment output without any sys.stdout trickery
#   (no LiveFileWriter, no stdout swap). The stored list is then passed to all
#   three output writers.

succeeded_paths = set()
failed_files    = {}
n = len(audio_files)

for i, audio_path in enumerate(audio_files, 1):
    raw_stem    = os.path.splitext(os.path.basename(audio_path))[0]
    output_stem = unicodedata.normalize("NFC", raw_stem)
    txt_path    = os.path.join(OUTPUT_DIR, output_stem + ".txt")
    srt_path    = os.path.join(OUTPUT_DIR, output_stem + ".srt")
    vtt_path    = os.path.join(OUTPUT_DIR, output_stem + ".vtt")

    print(f"\n[{i}/{n}] Processing: {os.path.basename(audio_path)}")

    # --- Audio preparation ---------------------------------------------------
    # For video files: strip video stream + resample to 16 kHz mono WAV.
    # For audio files: resample to 16 kHz mono WAV (if PRECONVERT_AUDIO=True).
    # Files already in optimal WAV format are detected and passed through as-is.
    # Keeping the model loaded ("hot") between files avoids re-initialization
    # overhead and is the most impactful single architectural optimisation.
    _temp_audio = None
    transcribe_path = audio_path
    _is_video = os.path.splitext(audio_path)[1].lower() in VIDEO_EXTENSIONS

    if _is_video or PRECONVERT_AUDIO:
        _label = "Video — extracting audio" if _is_video else "Pre-converting to 16 kHz mono WAV"
        print(f"  {_label}...")
        try:
            _prepared, _converted = _prepare_audio(audio_path, is_video=_is_video)
            if _converted:
                _temp_audio     = _prepared
                transcribe_path = _prepared
                print(f"  Audio ready.")
            else:
                print(f"  Already 16 kHz mono WAV — skipping conversion.")
        except RuntimeError as e:
            msg = str(e)
            print(f"[FAIL] {os.path.basename(audio_path)}: {msg}")
            failed_files[audio_path] = msg
            continue

    # .txt is NOT pre-cleared — if transcription fails, the previous successful
    # .txt is preserved. _write_timestamped_txt opens with "w", truncating only
    # on a successful write.

    segments = None
    _t0 = time.time()

    try:
        # model.transcribe() returns (lazy_generator, TranscriptionInfo).
        # Transcription runs as we iterate the generator. Consuming it here
        # prints segments in real time and stores them for the writers below.
        segments_gen, info = model.transcribe(
            transcribe_path,
            word_timestamps=True,
            **_options,
        )

        # Language detection result from the first ~30s of audio
        print(f"  Language: {info.language} ({info.language_probability:.0%})")

        # VAD stats: how much audio was skipped as non-speech
        if VAD_FILTER and info.duration > 0:
            _pct = (1 - info.duration_after_vad / info.duration) * 100
            print(
                f"  VAD: {info.duration:.1f}s total "
                f"\u2192 {info.duration_after_vad:.1f}s speech "
                f"({_pct:.0f}% filtered)"
            )

        # Consume the lazy generator — each iteration decodes one segment and
        # prints it immediately, giving real-time transcript output.
        segments = []
        for seg in segments_gen:
            ts_s = format_timestamp(seg.start, always_include_hours=False)
            ts_e = format_timestamp(seg.end,   always_include_hours=False)
            print(f"[{ts_s} --> {ts_e}] {seg.text.strip()}")
            segments.append(seg)

    except RuntimeError as e:
        msg = str(e)
        if "out of memory" in msg.lower():
            msg = (
                "GPU out of memory. "
                "Try a shorter audio file, or restart the runtime to free VRAM: "
                "Runtime > Restart session."
            )
        else:
            msg = f"RuntimeError: {e}"
        failed_files[audio_path] = msg
        print(f"\n[FAIL] {os.path.basename(audio_path)}: {msg}")

    except Exception as e:
        msg = str(e)
        failed_files[audio_path] = msg
        print(f"\n[FAIL] {os.path.basename(audio_path)}: {msg}")

    finally:
        if _temp_audio and os.path.exists(_temp_audio):
            os.unlink(_temp_audio)

    if segments is None:
        continue

    if not segments:
        print(f"  No speech detected in: {os.path.basename(audio_path)}")

    succeeded_paths.add(audio_path)

    # Write .txt from word-aligned segment data (identical format to the
    # openai-whisper AltTimestampMethod variant).
    if segments:
        _write_timestamped_txt(segments, txt_path)
        print(f"  Saved: {output_stem}.txt")

    print(f"  Completed in {time.time() - _t0:.1f}s")

    # Write .srt and .vtt using custom writers (get_writer not available in
    # faster-whisper). The segments list already has word-aligned boundaries.
    for _writer_fn, _fmt, _fpath in [
        (_write_srt, "srt", srt_path),
        (_write_vtt, "vtt", vtt_path),
    ]:
        try:
            _writer_fn(segments, _fpath)
            print(f"  Saved: {output_stem}.{_fmt}")
        except Exception as e:
            print(f"  [WARN] Could not write .{_fmt} for '{output_stem}': {e}")

# ── Phase 11: Zip All Transcribed Files ──────────────────────────────────────

if not succeeded_paths:
    print("\n[ERROR] No files transcribed successfully. Nothing to zip.")
else:
    _files_to_zip = sorted([
        os.path.join(OUTPUT_DIR, fname)
        for fname in os.listdir(OUTPUT_DIR)
        if os.path.splitext(fname)[1].lower() in TRANSCRIBED_EXTS
    ])

    if not _files_to_zip:
        print("\n[WARN] Output directory has no .txt/.srt/.vtt files to zip.")
    else:
        with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
            for fpath in _files_to_zip:
                zf.write(fpath, arcname=os.path.basename(fpath))

        _zip_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
        print(
            f"\nZipped {len(_files_to_zip)} file(s) -> {ZIP_PATH} "
            f"({_zip_mb:.2f} MB)"
        )
        print(
            "REMINDER: Download your zip before the Colab session ends. "
            "/content/ is erased when the runtime disconnects."
        )

# ── Phase 12: Final Summary ───────────────────────────────────────────────────

print("\n" + "=" * 52)
print("TRANSCRIPTION SUMMARY")
print(f"  Total files : {n}")
print(f"  Succeeded   : {len(succeeded_paths)}")
print(f"  Failed      : {len(failed_files)}")
if failed_files:
    print("\nFailed files:")
    for _path, _reason in failed_files.items():
        print(f"  x {os.path.basename(_path)}")
        print(f"    Reason: {_reason}")
print("=" * 52)
